# EC. Primer Parcial — Proyecto Práctico HPC
## Implementación secuencial

**Asignatura:** Cómputo de Alto Rendimiento  
**Integrante 1:** Elliot Gonzalez  

**Este notebook es la versión secuencial del programa, después, otro integrante podrá reutilizar esta base para construir la versión paralela.**


# Conceptos básicos de HPC

> **HPC (High Performance Computing)**. Son técnicas y recursos usados para resolver problemas que requieren bastante procesamiento.En este proyecto estudiaremos una idea fundamental: <br>
¿Podemos reducir el tiempo de ejecución dividiendo el trabajo entre varios workers? <br>
Para contestarlo necesitamos primero una referencia: la ejecución **secuencial**.

> **Ejecución secuencial** Las tareas se realizan una después de otra<br>
  **Ejecución paralela** Podemos dividir las tareas para realizarlas simultaneamente 



#  Dataset

> Usaremos **Covertype (Forest CoverType)**, un dataset público sobre cobertura forestal. Contiene aproximadamente: <br>
581,000 observaciones, 54 variables predictoras, Variables principalmente numéricas, Una variable objetivo que indica el tipo de cobertura

> Utilizaremos las variables numéricas como entrada para una operación matemática repetitiva. Así podemos concentrarnos en procesar muchos datos y comparar una ejecución secuencial con una ejecución paralela.

# Operación matemática

Para cada fila calcularemos:

$$
f(x) = \sum_{j=1}^{54} \left(\sqrt{x_j^2 + 1} + \sin(x_j)\right)
$$


In [5]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_covtype


## Hardware

El rendimiento depende de la computadora utilizada. Por eso documentaremos información básica del sistema.


In [6]:
import platform

print("Sistema operativo:", platform.system())
print("Versión:", platform.version())
print("Arquitectura:", platform.machine())
print("Procesador:", platform.processor())
print("Núcleos lógicos disponibles:", os.cpu_count())


Sistema operativo: Linux
Versión: #31-Ubuntu SMP PREEMPT_DYNAMIC Sat Aug  1 04:26:38 UTC 2026
Arquitectura: x86_64
Procesador: 
Núcleos lógicos disponibles: 16


# Análisis Inicial

In [7]:
# Descargar y cargar el dataset

print("Cargando Covertype...")
covtype = fetch_covtype(as_frame=True)

print("Dataset cargado correctamente.")


Cargando Covertype...
Dataset cargado correctamente.


In [8]:
X = covtype.data
y = covtype.target

print("Número de filas:", X.shape[0])
print("Número de características:", X.shape[1])
print("Tamaño de la variable objetivo:", y.shape[0])

display(X.head())


Número de filas: 581012
Número de características: 54
Tamaño de la variable objetivo: 581012


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39
0,2596.0,51.0,3.0,258.0,0.0,510.0,221.0,232.0,148.0,6279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2590.0,56.0,2.0,212.0,-6.0,390.0,220.0,235.0,151.0,6225.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2804.0,139.0,9.0,268.0,65.0,3180.0,234.0,238.0,135.0,6121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2785.0,155.0,18.0,242.0,118.0,3090.0,238.0,238.0,122.0,6211.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2595.0,45.0,2.0,153.0,-1.0,391.0,220.0,234.0,150.0,6172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
print("Tipos de datos:")
print(X.dtypes)

print("\nValores nulos:")
print(X.isnull().sum().sum())

print("\nMemoria aproximada de X:")
print(f"{X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Tipos de datos:
Elevation                             float64
Aspect                                float64
Slope                                 float64
Horizontal_Distance_To_Hydrology      float64
Vertical_Distance_To_Hydrology        float64
Horizontal_Distance_To_Roadways       float64
Hillshade_9am                         float64
Hillshade_Noon                        float64
Hillshade_3pm                         float64
Horizontal_Distance_To_Fire_Points    float64
Wilderness_Area_0                     float64
Wilderness_Area_1                     float64
Wilderness_Area_2                     float64
Wilderness_Area_3                     float64
Soil_Type_0                           float64
Soil_Type_1                           float64
Soil_Type_2                           float64
Soil_Type_3                           float64
Soil_Type_4                           float64
Soil_Type_5                           float64
Soil_Type_6                           float64
Soil_Type_7       

# Preparar los datos

Convertir el DataFrame de Pandas a un arreglo de NumPy.

In [10]:
X_np = X.to_numpy(dtype=np.float64)

print("Tipo:", type(X_np))
print("Forma:", X_np.shape)
print("Tipo numérico:", X_np.dtype)


Tipo: <class 'numpy.ndarray'>
Forma: (581012, 54)
Tipo numérico: float64


# Crear la función matemática

Separaremos el cálculo en una función que recibe **una sola fila** y devuelve un número.

In [11]:
def procesar_fila(fila):
    # Aplica la operación matemática a una sola fila.
    resultado = np.sqrt(fila**2 + 1.0) + np.sin(fila)
    return np.sum(resultado)


In [12]:

#Antes de procesar cientos de miles de filas, probamos la función con un solo registro.resultado_prueba = procesar_fila(X_np[0])
resultado_prueba = procesar_fila(X_np[0])

print("Resultado de la primera fila:")
print(resultado_prueba)

print("\nTipo del resultado:")
print(type(resultado_prueba))


Resultado de la primera fila:
10349.575412542694

Tipo del resultado:
<class 'numpy.float64'>


# Implementación secuencial

In [13]:
def procesar_secuencial(datos):
    # Crea un arreglo para guardar un resultado por cada fila.
    resultados = np.empty(datos.shape[0], dtype=np.float64)

    # Procesa cada fila una por una.
    for i in range(datos.shape[0]):
        resultados[i] = procesar_fila(datos[i])

    return resultados


>Usaremos `time.perf_counter()` para medir el intervalo de tiempo.


In [14]:
inicio = time.perf_counter()

resultados_secuenciales = procesar_secuencial(X_np)

fin = time.perf_counter()

tiempo_secuencial = fin - inicio

print(f"Tiempo de ejecución secuencial: {tiempo_secuencial:.4f} segundos")
print(f"Cantidad de resultados: {len(resultados_secuenciales)}")


Tiempo de ejecución secuencial: 3.0405 segundos
Cantidad de resultados: 581012


# Revisar los resultados

In [15]:
resumen = {
    "Mínimo": np.min(resultados_secuenciales),
    "Máximo": np.max(resultados_secuenciales),
    "Promedio": np.mean(resultados_secuenciales),
    "Desviación estándar": np.std(resultados_secuenciales),
}

for nombre, valor in resumen.items():
    print(f"{nombre}: {valor:.6f}")

print("\nPrimeros 10 resultados:")
print(resultados_secuenciales[:10])


Mínimo: 2710.825127
Máximo: 17256.998245
Promedio: 8403.326135
Desviación estándar: 2499.193851

Primeros 10 resultados:
[10349.57541254 10134.7916659  13243.66486567 13259.93130119
 10014.26137233  9779.78737852 10450.90279756 10343.77435249
 10503.02567575 10421.62524823]


## Benchmark de la ejecución secuencial

> Hasta ahora medimos una ejecución individual. Sin embargo,una sola medición no es suficiente para sacar conclusiones de rendimiento.Para nuestra línea base secuencial haremos 3 ejecuciones independientes.

La idea es:

```text
Prueba 1 ──┐
Prueba 2 ──┼──→ promedio
Prueba 3 ──┘
             ↓
        línea base
```


In [16]:
 #Ejecuta el procesamiento secuencial varias veces y mide cada ejecución.

def medir_secuencial(datos, repeticiones=3):
    tiempos = []

    for prueba in range(repeticiones):
        inicio = time.perf_counter()

        resultados = procesar_secuencial(datos)

        fin = time.perf_counter()

        tiempo = fin - inicio
        tiempos.append(tiempo)

        print(f"Prueba {prueba + 1}: {tiempo:.4f} segundos")

    return tiempos

# ejecutaremos la misma operación tres veces para ver si los tiempos son similares o si existe variación 
tiempos_secuenciales = medir_secuencial(X_np, repeticiones=3)

tiempo_promedio_secuencial = np.mean(tiempos_secuenciales)
desviacion_secuencial = np.std(tiempos_secuenciales, ddof=1)

print("\n--- Resumen ---")
print(f"Tiempo promedio: {tiempo_promedio_secuencial:.4f} segundos")
print(f"Desviación estándar: {desviacion_secuencial:.4f} segundos")



Prueba 1: 3.0211 segundos
Prueba 2: 2.5153 segundos
Prueba 3: 2.5336 segundos

--- Resumen ---
Tiempo promedio: 2.6900 segundos
Desviación estándar: 0.2869 segundos


###  Validar el benchmark

> Además de medir el tiempo, debemos comprobar que las ejecuciones siguen produciendo el mismo resultado. Compararemos la última ejecución del benchmark con la ejecución secuencial de referencia que calculamos anteriormente.

In [17]:
# Volvemos a ejecutar una vez para conservar el resultado de referencia.
resultados_benchmark = procesar_secuencial(X_np)

assert np.allclose(
    resultados_benchmark,
    resultados_secuenciales,
    rtol=1e-12,
    atol=1e-12
)

print("Los resultados del benchmark coinciden con la línea base.")


Los resultados del benchmark coinciden con la línea base.
